# Unit 9: Translation (German → English)

[Qwen2.5-VL](https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct) is a
vision-language model, but it is also a strong **text** chat model. Here we use
it **text-only** (no images) to translate the German transcript from
`00_PrepData` into English.

We translate **segment by segment** so each request stays short and the
timestamps are preserved, this also scales to clips of any length.

**Sources**
- Qwen2.5-VL: <https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct> ([report](https://arxiv.org/abs/2502.13923))
- Model class `QwenChat` lives in `src/video_pipeline/qwen_text.py`.

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

# Make the vendored `video_pipeline` package importable even if the project
# was not installed (e.g. running `jupyter` outside `uv run`).
SRC = Path.cwd() / "src"
if SRC.exists() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PREPARED = Path("prepared")
PREPARED.mkdir(exist_ok=True)

transcript = json.loads((PREPARED / "transcript.json").read_text(encoding="utf-8"))
print(f"loaded transcript: {len(transcript['segments'])} segments, "
      f"{len(transcript['text'])} chars")
print("\nfirst 300 chars (German):\n", transcript["text"][:300])

## Translate each segment with Qwen2.5-VL

`QwenChat.translate(text, source, target)` wraps a single system+user chat turn.
We load the model once (`with QwenChat() as qc:`) and reuse it across all
segments, then it is unloaded and the GPU cache is cleared on exit.

In [ ]:
from video_pipeline.qwen_text import QwenChat

translated_segments = []
with QwenChat() as qc:
    for i, seg in enumerate(transcript["segments"], start=1):
        english = qc.translate(seg["text"], source="German", target="English")
        translated_segments.append({**seg, "text": english})
        print(f"[{i}/{len(transcript['segments'])}] {english[:70]}")

english_text = "\n".join(s["text"] for s in translated_segments)

## Save a `TranslationResult`

We store the full English text in the project's `TranslationResult` schema so
the next notebook (and the CLI) can consume it.

In [ ]:
from video_pipeline.models import TranslationResult

result = TranslationResult(
    source_path=str(PREPARED / "transcript.json"),
    source_language="German",
    target_language="English",
    source_text=transcript["text"],
    text=english_text,
)
out = PREPARED / "translation.json"
out.write_text(result.model_dump_json(indent=2), encoding="utf-8")
print("wrote", out, "-", len(english_text), "chars")

In [ ]:
# German vs English, side by side per segment.
for de, en in zip(transcript["segments"][:6], translated_segments[:6]):
    print(f"[{de['start']:6.1f}s] DE: {de['text']}")
    print(f"          EN: {en['text']}\n")

---
**Next:** `02_Summarize_OCR.ipynb` — summarise the English transcript and OCR the
on-screen text from the video frames.